In [1]:
from data_collection.data_adapters import DatabentoGenericAdapter
from dotenv import load_dotenv
from pathlib import Path
import os
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
load_dotenv()
pl.enable_string_cache()

CACHE_DIR = Path.cwd().resolve().parents[0] / "data" / "dbn_cache" / "generic"
adapter = DatabentoGenericAdapter(os.getenv("DATABENTO-API-KEY"), CACHE_DIR)

print(f"CACHE_DIR: {CACHE_DIR}")
# Clear the cache if needed
# adapter.clear_cache()

with open("../top_250_universe.txt", "r") as f:
    ACTIVE_SET = [s for s in [line.strip() for line in f] if s not in ['CRCL', 'OMDA']]
START = "2023-06-01"
END = "2025-06-01"
DATASET = "XNAS.ITCH"

df_bars = (
    adapter.scan_data(
        tickers="ALL_SYMBOLS",
        start_date=START,
        end_date=END,
        dataset=DATASET,
        schema="ohlcv-1d"
    )
    .select(["ts_event", "symbol", "close"])
    .with_columns(
        pl.col("symbol").cast(pl.Categorical)
    )
    .filter(
        pl.col("symbol").is_in(ACTIVE_SET)
    )
    .sort(["ts_event", "symbol"])
)

CACHE_DIR: /Users/michaelcusack-nelkin/GitHub/Summer_Research_2026/data/dbn_cache/generic
[GenericAdapter] scan_parquet: 548cfb7cc98703d7d8a8a14fe79dfe1a.parquet


In [2]:
CAX = (
     pl.read_csv("../CAX/corporate_actions.csv")
     .sort(["date", "symbol"])
     .with_columns(
         pl.col("date").str.to_datetime(time_unit="ns", time_zone="UTC"),
         pl.col("symbol").cast(pl.Categorical)
     )
)
CAX

symbol,date,dividend,split
cat,"datetime[ns, UTC]",f64,f64
"""AGG""",2023-06-01 00:00:00 UTC,0.258,0.0
"""AJG""",2023-06-01 00:00:00 UTC,0.55,0.0
"""ALL""",2023-06-01 00:00:00 UTC,0.89,0.0
"""ANGL""",2023-06-01 00:00:00 UTC,0.131,0.0
"""BAC""",2023-06-01 00:00:00 UTC,0.22,0.0
…,…,…,…
"""FIS""",2026-06-11 00:00:00 UTC,0.44,0.0
"""FNV""",2026-06-11 00:00:00 UTC,0.44,0.0
"""JXN""",2026-06-11 00:00:00 UTC,0.9,0.0


In [3]:
daily_px = (
      df_bars
      .collect(engine="streaming")
    #   .filter(pl.col("symbol").is_in(ACTIVE_SET))
      .join(CAX, left_on=["ts_event", "symbol"], right_on=["date", "symbol"], how="left")
      .sort(["symbol", "ts_event"])
      .with_columns(
          pl.col("dividend").fill_null(0.0),
          # 0.0 / null == "no split" -> ratio of 1.0, not 0.0
          pl.when(pl.col("split").fill_null(0.0) == 0.0)
            .then(1.0).otherwise(pl.col("split")).alias("split"),
      )
      .with_columns(
          (pl.col("split") / (1 - pl.col("dividend") / pl.col("close").shift().over("symbol"))).alias("CAX")
      )
      .with_columns(pl.col("CAX").fill_null(1.0))            # first row per symbol: no prev_close
      .with_columns(pl.col("CAX").cum_prod().over("symbol"))
  )
daily_px

ts_event,symbol,close,dividend,split,CAX
"datetime[ns, UTC]",cat,f64,f64,f64,f64
2023-06-01 00:00:00 UTC,"""AAL""",14.66,0.0,1.0,1.0
2023-06-02 00:00:00 UTC,"""AAL""",14.84,0.0,1.0,1.0
2023-06-05 00:00:00 UTC,"""AAL""",14.79,0.0,1.0,1.0
2023-06-06 00:00:00 UTC,"""AAL""",14.91,0.0,1.0,1.0
2023-06-07 00:00:00 UTC,"""AAL""",15.17,0.0,1.0,1.0
…,…,…,…,…,…
2025-05-23 00:00:00 UTC,"""ZS""",253.99,0.0,1.0,1.0
2025-05-27 00:00:00 UTC,"""ZS""",255.9,0.0,1.0,1.0
2025-05-28 00:00:00 UTC,"""ZS""",255.52,0.0,1.0,1.0


In [4]:
adj_px = (
    daily_px
    .with_columns((pl.col("close") * pl.col("CAX")).alias("adj_close"))
    .select(["ts_event", "symbol", "adj_close"])
)
adj_px

ts_event,symbol,adj_close
"datetime[ns, UTC]",cat,f64
2023-06-01 00:00:00 UTC,"""AAL""",14.66
2023-06-02 00:00:00 UTC,"""AAL""",14.84
2023-06-05 00:00:00 UTC,"""AAL""",14.79
2023-06-06 00:00:00 UTC,"""AAL""",14.91
2023-06-07 00:00:00 UTC,"""AAL""",15.17
…,…,…
2025-05-23 00:00:00 UTC,"""ZS""",253.99
2025-05-27 00:00:00 UTC,"""ZS""",255.9
2025-05-28 00:00:00 UTC,"""ZS""",255.52


In [5]:
daily_ret = (
    adj_px
    .with_columns(
        pl.col("adj_close").pct_change().over("symbol").alias("ret")
    )
    .pivot(
        on="symbol",
        on_columns=ACTIVE_SET,
        index="ts_event",
        values="ret"
    )
    .with_columns(
        pl.all().exclude("ts_event").fill_null(pl.all().exclude("ts_event").mean())
    )
    .with_columns(
        (pl.all().exclude("ts_event") - pl.all().exclude("ts_event").mean()).round(6)
    )
)
daily_ret

ts_event,NVDA,TSLA,SPY,QQQ,AAPL,MSFT,AMZN,META,GOOGL,AVGO,PLTR,MSTR,GOOG,NFLX,TQQQ,TLT,AMD,IWM,COST,COIN,APP,MU,IBIT,SMCI,QCOM,TXN,CRWD,CSCO,INTC,HOOD,ADBE,SQQQ,BKNG,MRVL,AMAT,TMUS,…,SCHW,VRSK,DUOL,DPZ,AXP,SNOW,CSGP,SHV,NEE,DIS,RIVN,DHR,MCD,SHY,ACN,MDY,EWZ,SOXS,CDW,SLB,TSLQ,ENPH,FOXA,CAT,NTAP,UPST,CRDO,NARI,LPLA,WBD,SOUN,BSX,ARKK,ROKU,TGT,BECN,PM
"datetime[ns, UTC]",f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2023-06-01 00:00:00 UTC,-0.0,-0.0,0.0,0.0,0.0,0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,-0.0,0.0,0.0,-0.0,-0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,-0.0,0.0,0.0,0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.0,0.0,-0.0
2023-06-02 00:00:00 UTC,-0.021408,0.026902,0.013491,0.006259,0.00249,0.004946,0.006565,-0.004621,0.007726,0.043334,-0.008817,-0.000188,0.007765,-0.011219,0.018855,-0.012055,-0.020249,0.036361,0.00627,0.014123,0.004703,0.001497,-0.0,-0.042313,-0.003653,-0.003304,-0.044435,0.003118,0.008344,0.023769,0.023035,-0.029928,0.024091,-0.0215,0.000356,-0.063681,…,0.02794,0.003084,0.010247,0.025492,0.03332,0.031714,0.025504,0.000169,0.01403,0.017305,-0.011579,0.005462,0.011785,-0.002743,0.013964,0.031435,0.031295,0.019525,0.013071,0.038047,-0.021994,0.000746,0.020397,0.08185,-0.018177,-0.029261,-0.0148,-0.002791,0.041392,0.035078,0.011714,-0.002121,0.012606,0.021287,0.015974,0.038497,0.016286
2023-06-05 00:00:00 UTC,-0.009008,0.008421,-0.003998,-0.002275,-0.011323,0.000288,0.006263,-0.008301,0.00974,-0.014004,0.054365,-0.086325,0.010169,0.003448,-0.006002,0.00095,0.001576,-0.015992,0.008982,-0.098401,0.034828,-0.023388,-0.0,-0.004415,-0.027608,-0.032469,0.012681,0.002703,-0.053782,-0.033898,-0.006,-0.002291,0.019152,-0.040247,-0.014688,0.008408,…,0.002047,0.013984,0.030553,-0.012049,-0.00727,0.025191,0.001419,-0.00074,-0.006956,0.002872,-0.015146,-0.005862,-0.00509,0.000568,-0.022417,-0.011734,-0.002806,0.076511,-0.041061,-0.010577,-0.007384,0.006796,0.004931,-0.021128,-0.015515,-0.046989,-0.037734,-0.006007,-0.015374,-0.007158,-0.096318,-0.009504,0.006668,0.012158,-0.021942,-0.008413,-0.017807
2023-06-06 00:00:00 UTC,-0.008403,0.03989,0.003539,0.001448,-0.000836,-0.007403,0.010126,0.00153,0.009549,-0.021866,0.00906,0.072786,0.009495,-0.009208,0.004334,0.007228,0.059541,0.028644,-0.003343,-0.09896,-0.10432,0.000812,-0.0,0.058019,0.031883,0.001779,-0.00234,-0.00807,0.041037,0.002806,-0.002603,-0.013789,0.011442,0.029326,0.00315,-0.023548,…,0.003322,-0.006176,0.034105,-0.008797,0.025844,0.008077,0.003734,0.000169,-0.000343,0.013409,0.017642,0.007874,-0.013224,-0.000906,0.017419,0.020671,0.023246,-0.038509,0.011301,0.020975,-0.033743,0.002284,0.009552,0.020362,0.000162,0.107549,0.007662,-0.011195,-0.001363,0.04357,-0.035038,-0.009176,0.017081,0.033295,0.019537,0.047852,0.012936
2023-06-07 00:00:00 UTC,-0.029175,-0.010576,-0.005115,-0.018914,-0.006757,-0.032833,-0.041243,-0.034214,-0.037731,0.003366,-0.065435,-0.040639,-0.039038,-0.002475,-0.056331,-0.015617,-0.054513,0.018387,-0.012039,0.001993,-0.083974,-0.006868,-0.0,0.011281,-0.010701,0.012303,-0.031776,-0.001098,0.009376,0.015905,-0.034265,0.047089,-0.028892,-0.002328,0.009274,-0.014644,…,0.017928,-0.020803,-0.088122,-0.005929,-0.005171,-0.076106,-0.043905,-0.000286,0.010258,0.000102,-0.045382,0.014175,-0.010478,-0.000784,0.008445,0.014766,-0.005229,0.0186,0.009215,0.026307,0.01522,0.000313,0.013122,0.038731,0.011887,-0.088669,0.011317,-0.017652,0.033928,0.072227,-0.020676,-0.000749,-0.015031,0.077316,-0.008538,0.02055,0.008905
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-05-23 00:00:00 UTC,-0.01

In [6]:
# causal_vol = daily_ret.with_columns(pl.all().exclude("ts_event").pow(2).ewm_mean(alpha=0.06, adjust=True).sqrt().shift(1))
# st_ret = daily_ret.select(pl.all().exclude("ts_event")) / causal_vol.select(pl.all().exclude("ts_event"))
# st_ret


# floor the vol so a collapsed forecast can't produce a divide-by-~0
# (cross-sectional: e.g. 1st percentile of that day's vols, or a small abs floor)
causal_vol = daily_ret.with_columns(
    pl.all().exclude("ts_event").pow(2).ewm_mean(alpha=0.06, adjust=True).sqrt().shift(1)
)
VOL_FLOOR = 1e-3   # ~0.1%/day
st_ret = (
    daily_ret.select(pl.all().exclude("ts_event"))
    / causal_vol.select(pl.all().exclude("ts_event")).select(pl.all().clip(lower_bound=VOL_FLOOR))
).select(pl.all().clip(-10, 10))   # a >10σ standardized daily return is a data artifact

st_ret


NVDA,TSLA,SPY,QQQ,AAPL,MSFT,AMZN,META,GOOGL,AVGO,PLTR,MSTR,GOOG,NFLX,TQQQ,TLT,AMD,IWM,COST,COIN,APP,MU,IBIT,SMCI,QCOM,TXN,CRWD,CSCO,INTC,HOOD,ADBE,SQQQ,BKNG,MRVL,AMAT,TMUS,PEP,…,SCHW,VRSK,DUOL,DPZ,AXP,SNOW,CSGP,SHV,NEE,DIS,RIVN,DHR,MCD,SHY,ACN,MDY,EWZ,SOXS,CDW,SLB,TSLQ,ENPH,FOXA,CAT,NTAP,UPST,CRDO,NARI,LPLA,WBD,SOUN,BSX,ARKK,ROKU,TGT,BECN,PM
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
-10.0,10.0,10.0,6.259,2.49,4.946,6.565,-4.621,7.726,10.0,-8.817,-0.188,7.765,-10.0,10.0,-10.0,-10.0,10.0,6.27,10.0,4.703,1.497,-0.0,-10.0,-3.653,-3.304,-10.0,3.118,8.344,10.0,10.0,-10.0,10.0,-10.0,0.356,-10.0,8.266,…,10.0,3.084,10.0,10.0,10.0,10.0,10.0,0.169,10.0,10.0,-10.0,5.462,10.0,-2.743,10.0,10.0,10.0,10.0,10.0,10.0,-10.0,0.746,10.0,10.0,-10.0,-10.0,-10.0,-2.791,10.0,10.0,10.0,-2.121,10.0,10.0,10.0,10.0,10.0
-0.586075,0.435993,-0.412762,-0.506264,-6.333781,0.081103,1.328766,-2.502046,1.755922,-0.450116,8.588146,-10.0,1.824054,0.428069,-0.443374,0.109763,0.108406,-0.612587,1.995292,-9.704506,10.0,-10.0,-0.0,-0.145331,-10.0,-10.0,0.397493,1.207455,-8.977668,-1.986388,-0.362797,-0.106622,1.107287,-2.60733,-10.0,0.183901,0.832401,…,0.102045,6.315648,4.152962,-0.658337,-0.3039,1.106357,0.077495,-0.74,-0.690562,0.231161,-1.821914,-1.494841,-0.601574,0.288419,-2.235983,-0.519916,-0.124886,5.458002,-4.375438,-0.387207,-0.467615,6.796,0.336721,-0.359534,-1.188859,-2.236701,-3.551174,-2.997772,-0.517334,-0.284222,-10.0,-6.241179,0.736748,0.795515,-1.913213,-0.304386,-1.52292
-0.624055,2.445601,0.43479,0.375443,-0.121337,-2.589461,1.905485,0.272552,1.305882,-0.829662,0.276635,1.416811,1.261014,-1.355991,0.378503,1.035759,5.079902,1.243372,-0.517938,-1.673776,-4.99057,0.058228,-0.0,2.362836,1.924781,0.091623,-0.087556,-3.343956,1.267891,0.115031,-0.189143,-0.796055,0.636535,1.087223,0.360271,-0.635026,-2.709093,…,0.205482,-0.725722,1.783777,-0.537609,1.311489,0.341445,0.253332,0.169,-0.037725,1.323704,1.572433,1.674874,-1.776483,-0.559827,1.117689,1.063582,1.281922,-0.820991,0.441908,0.918464,-2.512615,0.561563,0.787531,0.416647,0.011594,3.292478,0.318921,-2.855282,-0.053294,2.106581,-0.607065,-1.585672,2.061553,2.335681,1.222358,2.101597,0.913348
-2.374099,-0.421797,-0.712416,-5.607284,-1.147544,-7.175981,-5.917776,-7.053381,-4.724116,0.133538,-2.311245,-0.700386,-4.810636,-0.328699,-5.622668,-2.215924,-1.666557,0.744459,-2.085711,0.027588,-1.462794,-0.577506,-0.0,0.305997,-0.489721,0.742308,-1.393067,-0.233817,0.268242,0.763154,-2.901946,2.865937,-1.756619,-0.084235,1.215202,-0.431731,-0.958217,…,1.290949,-2.620064,-3.647059,-0.403737,-0.239842,-3.694746,-3.453592,-0.286,1.323463,0.00917,-3.41494,2.466795,-1.1163,-0.53757,0.524289,0.74647,-0.2659,0.415507,0.408059,1.177421,0.723466,0.085369,1.142888,0.900905,0.998161,-1.412549,0.54242,-2.617973,1.555752,2.506716,-0.393876,-0.108831,-1.3198,3.640824,-0.501366,0.648813,0.643507
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
-0.400951,-0.185018,-0.445083,-0.537645,-0.997228,-0.414814,-0.435369,-0.544211,-0.777881,-0.401753,-0.047433,-2.020114,-0.787855,-0.267392,-0.558795,0.169457,-0.173723,-0.228726,-0.861658,-0.828469,-0.014069,-0.481423,-1.740198,-0.499992,-0.579086,-0.799101,0.667921,-0.309043,-0.69326,-0.56203,-0.770736,0.379933,-0.15557,-0.410263,-0.525021,0.27871,-0.296493,…,0.07615

In [7]:
null_columns = [
    col for col in st_ret.columns 
    if st_ret.select(pl.col(col).is_null().all()).item()
]
null_columns

[]

In [8]:
X = st_ret.drop_nulls().to_numpy()

In [9]:
# Best rank k approximation of the returns covariance matrix
U, s, Vt = np.linalg.svd(X)
k = 10
U_k, S_k, Vt_k = U[:, :k], np.diag(s[:k]), Vt[:k, :]
F = U_k
L = S_k @ Vt_k

# Residuals: what the market factor can't explain
# Adding a floor to epsilon volatility for illiquid names
eps = X - F @ L          # (T, p)
eps_var  = np.var(eps, axis=0)
EPS_FLOOR = 0.1
eps_var  = np.maximum(eps_var, EPS_FLOOR)
D = np.diag(eps_var)         # (2000, 2000) diagonal

# Factor covariance
Sigma_f = np.cov(F, rowvar=False)       # (10, 10)

# Model-implied return covariance: Σ_r = L^T Σ_f L + D
Sigma_r = L.T @ Sigma_f @ L + D         # (2000, 2000)


In [10]:
evals = np.linalg.eigvalsh(Sigma_r)

In [11]:
cond_n = np.abs(np.max(evals)) / np.abs(np.min(evals))
print(f"Condition Number: {cond_n}")

Condition Number: 923.9319101735393


In [12]:
eps_var

array([0.35082163, 0.54959804, 0.1       , 0.1       , 0.74125957,
       0.42206565, 0.52171058, 0.66256352, 0.70173822, 0.49933709,
       0.69579912, 0.55219373, 0.7032237 , 0.80583167, 0.1       ,
       0.33675621, 0.54159472, 0.16892483, 0.70689794, 0.42659376,
       0.91402557, 0.60353328, 0.66494176, 0.80979312, 0.43627127,
       0.43575774, 0.64591843, 1.03631012, 0.84358799, 0.57318716,
       0.83432075, 1.21919768, 0.76772764, 0.44897033, 0.33120673,
       0.73861766, 0.52448155, 0.1       , 0.77618843, 0.41384398,
       0.51835481, 0.34823943, 0.44379023, 0.68407838, 0.29543389,
       0.4772971 , 1.27513663, 0.77526397, 0.1       , 0.77264892,
       0.90427579, 0.75156938, 0.99765703, 0.39311349, 0.29279498,
       0.73523864, 1.10214996, 1.00709315, 0.87012003, 0.73805483,
       0.97819701, 0.308972  , 0.20865652, 0.29657618, 0.15630133,
       0.56708275, 0.1       , 0.1       , 0.92581509, 0.88775942,
       0.35410748, 0.73139828, 0.71804887, 0.70850056, 1.08302

In [13]:
d = np.diag(Sigma_r)
print("diag  min/median/max:", d.min(), np.median(d), d.max())   # want median ≈ 1, max single digits
print("cond:", np.linalg.cond(Sigma_r))                          # want O(1e1–1e3), not 1e7

diag  min/median/max: 0.10320227905178701 1.398678512184229 1.8932486214601283
cond: 923.931910173443


In [14]:
Sigma_r - (L.T @ Sigma_f @ L + D)

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(249, 249))

In [15]:
np.save("loadings.npy", L)
np.save("factor_returns.npy", F)
np.save("idiosyncratic_diag_vec.npy", eps_var)
np.save("factor_cov_matrix.npy", Sigma_f)
np.save("cov_matrix.npy", Sigma_r)


### Modeling eps (idiosyncratic) with GARCH(1,1)

In [16]:
L[0,:]

array([  7.998265  ,  16.14156779,  24.98527953,  21.72034352,
        13.41263151,  15.62066372,  16.38259919,  11.05071033,
        14.26060536,  16.96609651,   7.39402958,   9.65760962,
        14.2833682 ,   6.44697616,  23.35404095,  -0.37750309,
         9.26426107,  22.63100927,  12.9068647 ,  17.11379903,
        11.93651663,  14.43925315,   6.51216512,   5.69617175,
        13.55168079,  13.4731356 ,   7.30416618,  12.3575777 ,
        15.10569628,  16.93247004,  14.77718696, -16.62846542,
        16.91014834,   9.76446522,  15.13951223,   1.29715167,
         7.17486615,  15.30612323,  10.67225447,  14.02377061,
        18.25551208,  12.98401034,  11.38954184,  14.71739016,
        10.27715856,   1.78751747,   7.07980245,   9.31496627,
        24.97162172,  11.26712892,  10.24877835,   0.10806899,
        11.91147312,  14.73580424,  18.97982886,  15.34551248,
        15.07251247,   7.69477109,   5.02094218,  15.42333555,
        10.34972342,  17.54830973,  22.38480822,   4.56